
#### 2.7 Experiments: Problem (tokenizer_experiments): Experiments with tokenizers


In [76]:
vocab_tn = "results/tiny_stories/tiny_stories_vocab.pkl"
merges_tn = "results/tiny_stories/tiny_stories_merges.pkl"

vocab_owt = "results/owt/owt_train_vocab.pkl"
merges_owt = "results/owt/owt_train_merge.pkl"

special_tokens = ["<|endoftext|>"]

tn_filepath = "data/TinyStoriesV2-GPT4-train.txt"
owt_filepath = "data/owt_train.txt"


In [74]:
import pickle

def get_files(vocab_file, merge_file):
    with open(vocab_file, "rb") as f:
        vocab = pickle.load(f)
    with open(merge_file, "rb") as f:
        merges= pickle.load(f)
    return vocab, merges

In [89]:
from cs336_basics.tokenizer import Tokenizer

tokenizer_owt = Tokenizer.from_files(vocab_owt, merges_owt, special_tokens)
tokenizer_tn = Tokenizer.from_files(vocab_tn, merges_tn, special_tokens)

#### (a) Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively), encode these sampled documents into integer IDs. What is each tokenizer’s compression ratio (bytes/token)?
#### b) What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer? Compare the compression ratio and/or qualitatively describe what happens.

In [78]:
from typing import Iterator

class DocumentSampler:
    def __init__(self, file_path: str, delimiter: str = "<|endoftext|>", chunk_size: int = 1) -> None:
        self.file_path = file_path
        self.delimiter = delimiter
        self.chunk_size = chunk_size * 1024 *1024 # MB
    
    def sample_all(self) -> Iterator[str]:
        with open(self.file_path, "r", encoding="utf-8", errors="replace") as file:
            while True:
                chunk = file.read(self.chunk_size)
                if chunk == "":
                    break
                yield chunk
    
    def sample(self, num_samples: int = 10):
        buffer = ""
        chunk = ""
        docs_sampled = 0
        with open(self.file_path, "r", encoding="utf-8", errors="replace") as file:
            while docs_sampled < num_samples:
                chunk = file.read(self.chunk_size)
                if chunk == "":
                    break
                chunk = buffer + chunk
                parts = chunk.split(self.delimiter)

                buffer = parts[-1]
                docs = parts[:-1]

                for doc in docs:
                    if doc:
                        yield doc
                        docs_sampled += 1
                    if docs_sampled >= num_samples:
                        return

In [79]:
owt_sampler = DocumentSampler(owt_filepath)
tn_sampler = DocumentSampler(tn_filepath)

In [96]:
def get_compression_ratio(documents: list[str], tokenizer) -> dict:
    """Calculate compression metrics for a list of documents"""
    
    total_bytes = 0
    total_tokens = 0
    ratios = []
    
    for doc in documents:
        # Calculate bytes (UTF-8 encoding)
        doc_bytes = len(doc.encode('utf-8'))
        
        # Tokenize
        token_ids = tokenizer.encode(doc)
        doc_tokens = len(token_ids)
        
        # Individual document ratio
        doc_ratio = doc_bytes / doc_tokens if doc_tokens > 0 else 0
        ratios.append(doc_ratio)
        
        # Accumulate totals
        total_bytes += doc_bytes
        total_tokens += doc_tokens
    
    # Overall compression ratio
    overall_ratio = total_bytes / total_tokens if total_tokens > 0 else 0
    
    return {
        'overall_ratio': overall_ratio,
        'individual_ratios': ratios,
        'avg_ratio': sum(ratios) / len(ratios) if ratios else 0,
        'total_bytes': total_bytes,
        'total_tokens': total_tokens
    }

# Usage for your experiment:
def analyze_compression():
    # Sample documents
    owt_docs = list(owt_sampler.sample(20))
    tn_docs = list(tn_sampler.sample(20))
    
    print("=== OpenWebText Documents ===")
    
    # Test OWT tokenizer on OWT data
    owt_on_owt = get_compression_ratio(owt_docs, tokenizer_owt)
    print(f"OWT tokenizer on OWT data: {owt_on_owt['overall_ratio']:.2f} bytes/token")
    
    # Test TN tokenizer on OWT data  
    tn_on_owt = get_compression_ratio(owt_docs, tokenizer_tn)
    print(f"TN tokenizer on OWT data: {tn_on_owt['overall_ratio']:.2f} bytes/token")
    
    print("\n=== TinyStories Documents ===")
    
    # Test OWT tokenizer on TN data
    owt_on_tn = get_compression_ratio(tn_docs, tokenizer_owt)
    print(f"OWT tokenizer on TN data: {owt_on_tn['overall_ratio']:.2f} bytes/token")
    
    # Test TN tokenizer on TN data
    tn_on_tn = get_compression_ratio(tn_docs, tokenizer_tn)
    print(f"TN tokenizer on TN data: {tn_on_tn['overall_ratio']:.2f} bytes/token")
    
    return {
        'owt_on_owt': owt_on_owt,
        'tn_on_owt': tn_on_owt,
        'owt_on_tn': owt_on_tn,
        'tn_on_tn': tn_on_tn
    }
tokenizer_owt = Tokenizer.from_files(vocab_owt, merges_owt, special_tokens)
tokenizer_tn = Tokenizer.from_files(vocab_tn, merges_tn, special_tokens)
results = analyze_compression()

=== OpenWebText Documents ===
OWT tokenizer on OWT data: 4.46 bytes/token
TN tokenizer on OWT data: 3.15 bytes/token

=== TinyStories Documents ===
OWT tokenizer on TN data: 3.97 bytes/token
TN tokenizer on TN data: 4.06 bytes/token


##### They both encode about 4 bytes per token for their respective datasets. 
##### The tinystories tokenizer drops in performance in compression ratio when used on the Openwebtext dataset. This intuitively makes sense, because there are more unique tokens it hasn't probably seen in this dataset.

#### c) Estimate the throughput of your tokenizer (e.g., in bytes/second). How long would it take to tokenize the Pile dataset (825GB of text)?

In [97]:
import time
import statistics

def estimate_tokenizer_throughput(tokenizer, sample_sizes=[1000, 10000, 100000], num_runs=3):
    """
    Estimate tokenizer throughput in bytes/second
    
    Args:
        tokenizer: Your tokenizer object
        sample_sizes: List of text sizes to test (in characters)
        num_runs: Number of runs per sample size for averaging
    """
    
    print("=== TOKENIZER THROUGHPUT ESTIMATION ===\n")
    
    # Get sample data
    with open("data/owt_train.txt", "r", encoding="utf-8") as f:
        full_text = f.read(max(sample_sizes) * 2)  # Read enough for largest sample
    
    throughputs = []
    
    for size in sample_sizes:
        print(f"Testing with {size:,} characters...")
        
        # Extract sample text
        sample_text = full_text[:size]
        sample_bytes = len(sample_text.encode('utf-8'))
        
        times = []
        
        for run in range(num_runs):
            start_time = time.perf_counter()
            tokens = tokenizer.encode(sample_text)
            end_time = time.perf_counter()
            
            elapsed = end_time - start_time
            times.append(elapsed)
        
        # Calculate statistics
        avg_time = statistics.mean(times)
        throughput = sample_bytes / avg_time  # bytes/second
        throughputs.append(throughput)
        
        print(f"  Size: {sample_bytes:,} bytes")
        print(f"  Tokens: {len(tokens):,}")
        print(f"  Time: {avg_time:.4f}s ± {statistics.stdev(times):.4f}s")
        print(f"  Throughput: {throughput:,.0f} bytes/second")
        print(f"  Throughput: {throughput/1024/1024:.2f} MB/second")
        print()
    
    # Overall estimate
    overall_throughput = statistics.mean(throughputs)
    
    print("=== SUMMARY ===")
    print(f"Estimated throughput: {overall_throughput:,.0f} bytes/second")
    print(f"Estimated throughput: {overall_throughput/1024/1024:.2f} MB/second")
    
    # Estimate time for The Pile
    pile_bytes = 825 * 1024 * 1024 * 1024  # 825 GB
    pile_time_seconds = pile_bytes / overall_throughput
    pile_time_hours = pile_time_seconds / 3600
    pile_time_days = pile_time_hours / 24
    
    print(f"\n=== THE PILE ESTIMATION (825 GB) ===")
    print(f"Time to tokenize: {pile_time_seconds:,.0f} seconds")
    print(f"Time to tokenize: {pile_time_hours:.1f} hours")
    print(f"Time to tokenize: {pile_time_days:.1f} days")
    
    return {
        'throughput_bytes_per_sec': overall_throughput,
        'throughput_mb_per_sec': overall_throughput/1024/1024,
        'pile_time_seconds': pile_time_seconds,
        'pile_time_hours': pile_time_hours,
        'pile_time_days': pile_time_days
    }

tokenizer_owt = Tokenizer.from_files(vocab_owt, merges_owt, special_tokens)
tokenizer_tn = Tokenizer.from_files(vocab_tn, merges_tn, special_tokens)
# Test both tokenizers
print("OpenWebText Tokenizer:")
owt_results = estimate_tokenizer_throughput(tokenizer_owt)

print("\n" + "="*60 + "\n")

print("TinyStories Tokenizer:")
tn_results = estimate_tokenizer_throughput(tokenizer_tn)

OpenWebText Tokenizer:
=== TOKENIZER THROUGHPUT ESTIMATION ===

Testing with 1,000 characters...
  Size: 1,006 bytes
  Tokens: 224
  Time: 0.0002s ± 0.0001s
  Throughput: 4,872,402 bytes/second
  Throughput: 4.65 MB/second

Testing with 10,000 characters...
  Size: 10,058 bytes
  Tokens: 2,179
  Time: 0.0016s ± 0.0007s
  Throughput: 6,265,217 bytes/second
  Throughput: 5.97 MB/second

Testing with 100,000 characters...
  Size: 101,048 bytes
  Tokens: 22,615
  Time: 0.0179s ± 0.0081s
  Throughput: 5,641,172 bytes/second
  Throughput: 5.38 MB/second

=== SUMMARY ===
Estimated throughput: 5,592,930 bytes/second
Estimated throughput: 5.33 MB/second

=== THE PILE ESTIMATION (825 GB) ===
Time to tokenize: 158,385 seconds
Time to tokenize: 44.0 hours
Time to tokenize: 1.8 days


TinyStories Tokenizer:
=== TOKENIZER THROUGHPUT ESTIMATION ===

Testing with 1,000 characters...
  Size: 1,006 bytes
  Tokens: 270
  Time: 0.0003s ± 0.0003s
  Throughput: 3,543,023 bytes/second
  Throughput: 3.38 MB/s